## 1. Distillation of the labelled 85k dataset.

### Necessary libraries

In [4]:
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import Counter
import os

### Distillation (Canonicalization)

In [8]:
print("Beginning of distilization process.")
data_path = os.path.join(os.getcwd(), "raw_entities.csv")
df = pd.read_csv(data_path, encoding="utf-8")


def normalize_entity(text, label):
    text = str(text).lower().strip()

    if label == "SKILL":
        if text in ["react", "reactjs", "react.js", "react js"]: return "react"
        if text in ["pl/sql", "plsql", "pl", "pl sql"]: return "pl/sql"
        if text in ["machine learning", "ml"]: return "machine learning"
        if text in ["deep learning", "dl"]: return "deep learning"
        if text in ["py", "python 3", "python 2", "python"]: return "python"
        if text in ["js", "javascript", "es5", "es6", "es7"]: return "javascript"
        if text in ["ai", "artificial intelligence"]: return "artificial intelligence"
        if text in ["postgres", "postgre", "postgresql"]: return "postgresql"
        if text in ["my sql", "mysql", "mysql workbench"]: return "mysql"
    return text

print("starting data normalization...")
print(f"data entity count prior distillization is: {len(df)}")
df["distilled_text"] = df.apply(lambda x: normalize_entity(x["text"], x["label"]), axis=1)

print("cleaning the text")
df = df[df["distilled_text"].str.len() > 1]

print(f"after cleaning the dataset we now left with {len(df)} entities")
print("distillation is complete")

Beginning of distilization process.
starting data normalization...
data entity count prior distillization is: 4107206
cleaning the text
after cleaning the dataset we now left with 4098044 entities
distillation is complete


### Creating the knowledge graph.

In [9]:
print("creating edges of the graph")
edges = []
node_counts = Counter()

grouped = df.groupby("doc_id")

for doc_id, group in grouped:
    entities = set(zip(group["distilled_text"], group["label"]))

    for text, label in entities:
        node_counts[(text, label)] += 1

    skills = sorted([t for t, l in entities if l == "SKILL"])
    titles = sorted([t for t, l in entities if l == "TITLE"])
    orgs = sorted([t for t, l in entities if l == "ORG"])

    for s1, s2 in combinations(skills, 2):
        edges.append({"source": s1, "target": s2, "type": "CO_SKILL"})

    for title in titles:
        for skill in skills:
            edges.append({"source": title, "target": skill, "type": "REQUIRES_SKILL"})
    
    for org in orgs:
        for skill in skills:
            edges.append({"source": org, "target": skill, "type": "HIRES_FOR"})
    
print("aggregating and filtering")
df_edges = pd.DataFrame(edges)

df_edges = df_edges.groupby(["source", "target", "type"]).size().reset_index(name="weight")

df_edges = df_edges[df_edges["weight"] > 5]

df_nodes = pd.DataFrame([
    {"id": text, "type": label, "count": count}
    for (text, label), count in node_counts.items()
])

valid_nodes = set(df_edges["source"]).union(set(df_edges["target"]))
df_nodes = df_nodes[df_nodes["id"].isin(valid_nodes)]

df_nodes.to_csv("kg_nodes.csv", index=False)
df_edges.to_csv("kg_edges.csv", index=False)

print(f"Knowledge graph built")
print(f"Nodes: {len(df_nodes)}")
print(f"Edges: {len(df_edges)}")
print("Files saved as kg_nodes and kg_edges")

creating edges of the graph
aggregating and filtering
Knowledge graph built
Nodes: 33965
Edges: 1683635
Files saved as kg_nodes and kg_edges
